# Which Investor Behaviors Encode Rate Sensitivity?

This notebook compares embeddings learned from stable stock pickers, tactical active funds, benchmark rebalancers, and passive-like funds. Every behavioral embedding is added to the same six characteristics and evaluated on permanent test firms under q−1 and q−2 timing.

## 1. Setup

Penalties are selected only in 2010–2016. Preprocessing and target clipping remain training-only within each quarter.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

def locate_root():
    start=Path.cwd().resolve()
    for candidate in [start,*start.parents]:
        if (candidate/'economic_shock_sensitivity').exists() and (candidate/'active_passive_extension').exists(): return candidate
    raise FileNotFoundError('Could not locate Asset_Embeddings root.')
ROOT=locate_root(); MODULE=ROOT/'economic_shock_sensitivity'; RESULTS=MODULE/'results'/'behavioral_rate_models'; RESULTS.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(MODULE/'src')); import embedding_rate_models as erm
robustness_gate=pd.read_parquet(MODULE/'results'/'incremental_robustness'/'incremental_robustness_gate.parquet'); display(robustness_gate); assert robustness_gate.passed.all()
targets=pd.read_parquet(MODULE/'data'/'processed'/'rolling_rate_targets'/'all_rolling_rate_targets.parquet'); targets=targets[(targets.specification.eq('stmt_ar00'))&targets.eligible]
quarters=sorted(targets.anchor_quarter.unique()); development=[q for q in quarters if int(q[:4])<=2016]


## 2. Build behavioral model inventory

RS-Binary and RS-L-Min are each fitted for four investor-behavior groups. Characteristics are included in every model so the comparison measures incremental information from each holdings geometry.

In [ ]:
styles=['stable_stock_picker','tactical_active','benchmark_rebalancer','passive_like']; constructions=['rs_binary','rs_l_min']; specs=[]
for lag,timing in [(1,'economic_q_minus_1'),(2,'public_q_minus_2')]:
    for style in styles:
        for construction in constructions:
            specs.append({'name':f'{style}__{construction}__{timing}','style':style,'construction':construction,'lag':lag,'timing':timing})
display(pd.DataFrame(specs))


## 3. Load quarterly matrices and select penalties

All design matrices are cached in memory. The selected penalty maximizes pooled validation variance R² over development quarters.

In [ ]:
datasets={}; inventory=[]
for spec in specs:
    for quarter in quarters:
        frame,features,input_quarter=erm.load_behavioral_frame(ROOT,targets,quarter,spec['style'],spec['construction'],spec['lag'],True)
        datasets[(spec['name'],quarter)]=(frame,features); inventory.append({'model':spec['name'],'quarter':quarter,'input_quarter':input_quarter,'firms':len(frame),'features':len(features)})
inventory=pd.DataFrame(inventory); inventory.to_parquet(RESULTS/'input_inventory.parquet',index=False)
display(inventory.groupby('model').agg(quarters=('quarter','nunique'),minimum_firms=('firms','min'),median_firms=('firms','median'),features=('features','max')).reset_index())
alpha_grid=[0.01,0.1,1,10,100,1000,10000]; cv_rows=[]
for number,spec in enumerate(specs,1):
    for alpha in alpha_grid:
        summaries=[]
        for quarter in development:
            frame,features=datasets[(spec['name'],quarter)]; _,summary=erm.fit_predict(frame,features,alpha,('train',),'validation'); summaries.append(summary)
        summaries=pd.DataFrame(summaries); cv_rows.append({'model':spec['name'],'alpha':alpha,'pooled_validation_r2':erm.pooled_variance_r2(summaries),'mean_correlation':summaries.correlation.mean()})
    print(f'{number:02d}/{len(specs)} {spec["name"]}')
cv=pd.DataFrame(cv_rows); selected=(cv.sort_values(['model','pooled_validation_r2'],ascending=[True,False]).groupby('model',as_index=False).first()[['model','alpha','pooled_validation_r2']].rename(columns={'alpha':'selected_alpha'}))
cv.to_parquet(RESULTS/'development_alpha_curves.parquet',index=False); selected.to_parquet(RESULTS/'selected_penalties.parquet',index=False); display(selected.sort_values('pooled_validation_r2',ascending=False))


## 4. Permanent test-firm evaluation

Models are refitted on train plus validation firms and predict only permanent test firms. Results are pooled separately over development, validation, and final-test quarters.

In [ ]:
alpha_lookup=selected.set_index('model').selected_alpha.to_dict(); predictions=[]; fit_rows=[]
for number,spec in enumerate(specs,1):
    for quarter in quarters:
        frame,features=datasets[(spec['name'],quarter)]; prediction,summary=erm.fit_predict(frame,features,alpha_lookup[spec['name']],('train','validation'),'test')
        prediction['model']=spec['name']; predictions.append(prediction); summary.update(model=spec['name'],style=spec['style'],construction=spec['construction'],timing=spec['timing'],quarter=quarter); fit_rows.append(summary)
    print(f'{number:02d}/{len(specs)} {spec["name"]}')
predictions=pd.concat(predictions,ignore_index=True); fits=pd.DataFrame(fit_rows); fits['time_sample']=np.select([fits.quarter.str[:4].astype(int).le(2016),fits.quarter.str[:4].astype(int).le(2020)],['development','validation'],default='final_test')
predictions.to_parquet(RESULTS/'permanent_test_predictions.parquet',index=False); fits.to_parquet(RESULTS/'quarterly_metrics.parquet',index=False)
summary=[]
for keys,group in fits.groupby(['style','construction','timing','time_sample']):
    summary.append({'style':keys[0],'construction':keys[1],'timing':keys[2],'time_sample':keys[3],'quarters':len(group),'pooled_variance_r2':erm.pooled_variance_r2(group),'mean_quarter_r2':group.variance_r2.mean(),'mean_correlation':group.correlation.mean()})
summary=pd.DataFrame(summary); summary.to_parquet(RESULTS/'behavioral_model_summary.parquet',index=False); display(summary.sort_values(['time_sample','timing','pooled_variance_r2'],ascending=[True,True,False]))


## 5. Compare behavioral groups with the all-S12 benchmark

The all-S12 RS-L-Min plus-characteristics result from Notebook 04 is included as the reference. Because behavioral samples can differ slightly, this table is descriptive; the primary claim remains the all-S12 result.

In [ ]:
all_s12=pd.read_parquet(MODULE/'results'/'ex_ante_embedding_models'/'target_recovery_summary.parquet')
reference=all_s12[all_s12.model.str.startswith('rs_l_min_10_plus_chars')][['model','time_sample','pooled_variance_r2']].copy(); reference['style']='all_s12'; reference['construction']='rs_l_min'; reference['timing']=reference.model.str.split('__').str[-1]
comparison=pd.concat([summary[['style','construction','timing','time_sample','pooled_variance_r2']],reference[['style','construction','timing','time_sample','pooled_variance_r2']]],ignore_index=True)
final_ranking=comparison[comparison.time_sample.eq('final_test')].sort_values(['timing','pooled_variance_r2'],ascending=[True,False]); final_ranking.to_parquet(RESULTS/'final_test_behavioral_ranking.parquet',index=False); display(final_ranking)


## 6. Interpretation gate

This comparison is complete if all behavioral models cover 60 quarters and produce finite final-test metrics. The ranking is an empirical result rather than a pass/fail hypothesis.

In [ ]:
gate=pd.DataFrame([{'test':'all 16 models cover 60 quarters','value':int((fits.groupby('model').quarter.nunique()==60).sum()),'passed':(fits.groupby('model').quarter.nunique()==60).all()},{'test':'all final-test pooled R2 values finite','value':int(np.isfinite(summary.loc[summary.time_sample.eq('final_test'),'pooled_variance_r2']).sum()),'passed':np.isfinite(summary.loc[summary.time_sample.eq('final_test'),'pooled_variance_r2']).all()}])
gate.to_parquet(RESULTS/'behavioral_comparison_gate.parquet',index=False); display(gate)
print('STOP: inspect the behavioral ranking before the post-shock holdings-adjustment test.')
